# ANALÍTICA EMPRESARIAL INTEGRADA
## LABORATORIO CALIFICADO N.° 07 — Auditoría de un modelo y asistente Text-to-SQL

**Semanas 13 y 14 · TECSUP 2026-II**
**Docente:** PILAR ROCIO SAYAN MEJIA
**Dataset real:** Default of Credit Card Clients — UCI Machine Learning Repository (ID 350): 30 000 clientes de tarjeta de crédito de un banco de Taiwán
**Objetivo:** decidir si un modelo de riesgo de crédito y un asistente que escribe SQL pueden usarse en el área de riesgos, con auditoría por subgrupos, explicabilidad, controles de gobierno, evaluación de seguridad y una recomendación integrada.

> Esta es una evaluación **grupal** de 16 puntos (más 4 individuales). La **introducción** de este notebook — instalación, parámetros, obtención de los datos y trazabilidad — ya está programada. A partir del Ejercicio 1, **el equipo escribe su propio código** y responde cada pregunta.

## Trabajo grupal y entrega

Ejecute el notebook en **Google Colab** de inicio a fin. Las respuestas, el informe y la recomendación quedan en este mismo archivo.

**Al final de la sesión:**
1. Suba este notebook, ejecutado y con todas las respuestas, a la carpeta `lc7/` del repositorio del equipo en GitHub.
2. Conserve la versión en el Colab del equipo y compártala con permiso de lectura.
3. Entregue **ambos URLs** en la actividad de la semana.

**Identificación del equipo**
- Uso de IA declarado (herramienta y partes del trabajo): ChatGPT ayudó a incorporar la lista de integrantes.

| N.° | Nombres y apellidos | Código | Aporte principal |
|---|---|---|---|
| 1 | Edwar Chuquiyuri Valdivia |  |  |
| 2 | Juan Diego Cuya Vera |  |  |
| 3 | Jesús Guzmán Livia |  |  |
| 4 | Christopher Rojas Zuñiga |  |  |

## Normas y buenas prácticas

- Conserve la **trazabilidad** y distinga datos observados, supuestos académicos y resultados simulados.
- **No cambie** los parámetros, los umbrales ni las reglas de decisión después de conocer la evaluación final para favorecer una conclusión.
- Sexo, edad y estado civil se usan **solo para auditar**: no son causas ni una recomendación para decidir créditos.
- La **clave** del proveedor de LLM nunca se escribe en el notebook: use los secretos de Colab.
- Se permite usar asistentes de IA **declarándolo** en la identificación del equipo. Todo comentario, respuesta e interpretación debe estar **escrito con palabras propias del equipo**. Texto de IA pegado sin revisar ni reformular, o un integrante que no puede explicar el trabajo: **descuento mínimo de 5 puntos** sobre el puntaje grupal.
- Puede reutilizar el trabajo de los laboratorios dirigidos 13 y 14, declarando su origen y adaptándolo a estos datos.
- La docente puede pedir a cualquier integrante que sustente el código y los resultados. La documentación vale 3 de los 16 puntos grupales.
- **Duración:** 190 minutos, con 15 minutos de receso.

# 1. Pregunta de negocio

**El área de riesgos de un banco quiere saber si puede usar un modelo de riesgo de crédito y un asistente que responde preguntas escribiendo SQL: ¿qué evidencia respalda cada uso y qué condiciones faltan?**

### Diseño del laboratorio
| Elemento | Definición |
|---|---|
| Base de datos | Default of Credit Card Clients (UCI, ID 350): una fila por cliente |
| Variable objetivo | `incumple`: 1 si el cliente no pagó el mes siguiente |
| Parte A — modelo | Regresión logística sin `sex`, `age` ni `marriage`; auditoría por subgrupos, SHAP y checklist |
| Parte B — asistente | Text-to-SQL con LLM sobre una tabla `credito` en DuckDB de solo lectura |
| Partición | 70 % entrenamiento y 30 % prueba, estratificada, semilla 2026 |

### Parámetros académicos
La finalidad, los umbrales y los criterios de aprobación **no provienen del archivo**: son supuestos de evaluación definidos en la celda de parámetros, que **no debe modificarse**.

In [1]:
# Instala en Colab las librerías que no vienen por defecto (xlrd lee el formato .xls de UCI)
%pip install -q xlrd openpyxl shap duckdb plotly scikit-learn

In [2]:
import hashlib
import os
import re
import sys
import warnings
from datetime import datetime, timezone
from io import BytesIO
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile

import duckdb
import numpy as np
import pandas as pd
import plotly.express as px
import shap
import sklearn

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)
VERSIONES = {"python": sys.version.split()[0], "pandas": pd.__version__, "numpy": np.__version__,
             "scikit-learn": sklearn.__version__, "shap": shap.__version__, "duckdb": duckdb.__version__}
print(VERSIONES)

{'python': '3.13.18', 'pandas': '2.2.3', 'numpy': '2.4.6', 'scikit-learn': '1.6.1', 'shap': '0.54.1', 'duckdb': '1.3.2'}


In [3]:
# ---------------- PARÁMETROS ACADÉMICOS DEL CASO (no modificar) ----------------
SEMILLA = 2026
np.random.seed(SEMILLA)

# Parte A — modelo y auditoría
PROP_PRUEBA = 0.30                # 70 % entrenamiento y 30 % prueba, estratificado por incumple
UMBRAL = 0.50                     # umbral de decisión; sensibilidad: 0.30 y 0.70
UMBRALES = [0.30, 0.50, 0.70]
SENSIBLES = ["sex", "age", "marriage"]            # solo para auditar; no entran al modelo
CORTES_EDAD = [0, 29, 44, 200]                    # grupos 21-29, 30-44 y 45 o más
ETIQUETAS_EDAD = ["21-29", "30-44", "45+"]
MIN_N = 100                       # grupos con menos personas se omiten por su incertidumbre
SHAP_N = 500                      # clientes de la prueba que se explican con SHAP
CONTROLES = ["Finalidad", "Minimización", "Calidad", "Linaje", "Acceso", "Transparencia", "Auditoría de subgrupos", "Supervisión humana"]

# Parte B — asistente Text-to-SQL
MAX_FILAS = 1000                  # tope de filas de cualquier consulta generada
print("Parámetros cargados.")

Parámetros cargados.


# 2. Obtención de los datos y trazabilidad

La celda siguiente descarga el **ZIP oficial de UCI (ID 350)** y lee la hoja de Excel. Si no lo logra, busca el archivo `default of credit card clients.xls` en la carpeta de la sesión. Si ninguna opción funciona, solicite una copia a la docente: **no sustituya la fuente ni genere datos ficticios.**

In [4]:
URL_UCI = "https://archive.ics.uci.edu/static/public/350/default+of+credit+card+clients.zip"
try:
    with urlopen(URL_UCI, timeout=120) as respuesta:
        with ZipFile(BytesIO(respuesta.read())) as z:
            nombre = next(n for n in z.namelist() if n.lower().endswith((".xls", ".xlsx")))
            df_raw = pd.read_excel(BytesIO(z.read(nombre)), header=1)
    fuente = "ZIP oficial UCI, ID 350"
except Exception as exc:
    print("Descarga no disponible:", type(exc).__name__)
    local = Path("default of credit card clients.xls")
    if not local.exists():
        raise FileNotFoundError("Descargue el archivo desde UCI (ID 350) y súbalo a Colab, o solicite una copia a la docente.")
    df_raw = pd.read_excel(local, header=1)
    fuente = "Archivo local descargado de UCI"
print("Fuente:", fuente)
print("Dimensión:", df_raw.shape)

Fuente: ZIP oficial UCI, ID 350
Dimensión: (30000, 25)


In [5]:
# Nombres de columna en minúsculas; pay_0 pasa a pay_1 para que la serie pay_1 ... pay_6 sea continua
df = df_raw.copy()
df.columns = [str(c).strip().lower().replace(" ", "_") for c in df.columns]
df = df.rename(columns={"pay_0": "pay_1", "default_payment_next_month": "incumple"})
OBLIGATORIAS = ["id", "limit_bal", "sex", "education", "marriage", "age", "pay_1", "pay_6", "bill_amt1", "pay_amt1", "incumple"]
faltan = [c for c in OBLIGATORIAS if c not in df.columns]
assert not faltan, f"Faltan columnas obligatorias: {faltan}"

# Trazabilidad: fuente, momento de la lectura, dimensiones, memoria y huella de los datos
TRAZA = {
    "fuente": fuente,
    "lectura_utc": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S"),
    "filas": len(df),
    "columnas": df.shape[1],
    "memoria_MB": round(float(df.memory_usage(deep=True).sum()) / 1e6, 2),
    "huella_filas": int(pd.util.hash_pandas_object(df, index=False).sum() % (10 ** 12)),
    "versiones": VERSIONES,
}
for k, v in TRAZA.items():
    print(f"{k}: {v}")
display(df.head())

fuente: ZIP oficial UCI, ID 350
lectura_utc: 2026-11-23 02:25:36
filas: 30000
columnas: 25
memoria_MB: 6.0
huella_filas: 193675118008
versiones: {'python': '3.13.18', 'pandas': '2.2.3', 'numpy': '2.4.6', 'scikit-learn': '1.6.1', 'shap': '0.54.1', 'duckdb': '1.3.2'}


,id,limit_bal,sex,education,marriage,age,pay_1,pay_2,pay_3,pay_4,pay_5,pay_6,bill_amt1,bill_amt2,bill_amt3,bill_amt4,bill_amt5,bill_amt6,pay_amt1,pay_amt2,pay_amt3,pay_amt4,pay_amt5,pay_amt6,incumple
0,1,20000,2,2,1,24,2,2,-1,-1,-2,-2,3913,3102,689,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,0,2,2682,1725,2682,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,0,0,29239,14027,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,0,0,46990,48233,49291,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,0,0,8617,5670,35835,20940,19146,19131,2000,36681,10000,9000,689,679,0


> **Desde aquí todo el código es del equipo**, incluida la conexión al LLM (puede adaptar la del laboratorio dirigido 14, declarándolo). Cada ejercicio tiene sus preguntas, una celda de código vacía para resolverlas y un espacio de **Respuesta** (máximo cinco líneas con cifras propias). Cada resultado se valora una sola vez dentro del criterio correspondiente de la rúbrica. El DataFrame `df` contiene los datos tal como se leyeron: no lo modifique; trabaje sobre copias y documente cada decisión.

# Ejercicio 1 — Ficha de datos, calidad y gobierno

### Pregunta 1.1

Registre la fuente, la fecha de descarga y las dimensiones. Construya la ficha de datos: finalidad, variable objetivo, propietario propuesto, variables sensibles solo para auditar, reglas de calidad y linaje. ¿Cuántos clientes hay y qué proporción incumple?

In [6]:
# Escriba aquí el código de su equipo y documente cada decisión con comentarios.

**Respuesta:**

### Pregunta 1.2

Cuantifique, sin eliminarlos, los códigos no definidos de education (0, 5 y 6), marriage (0) y pay_1 a pay_6 (−2 y 0), y los clientes duplicados al ignorar id. Decida qué hará con cada caso y justifique por qué conservarlos o agruparlos es una decisión de gobierno y no solo de limpieza.

In [7]:
# Escriba aquí el código de su equipo y documente cada decisión con comentarios.

**Respuesta:**

### Pregunta 1.3

Redacte el contrato de datos del archivo en cinco líneas: esquema, significado de las columnas, calidad esperada, propietario y condiciones de uso. ¿Qué dos afirmaciones no puede sostener con datos de un banco de Taiwán de 2005?

In [8]:
# Escriba aquí el código de su equipo y documente cada decisión con comentarios.

**Respuesta:**

> **Criterio de aceptación:** documenta cada decisión de calidad con su cifra, no elimina registros sin justificarlo y distingue lo que dice la fuente de lo que supone el equipo.

# Ejercicio 2 — Modelo y auditoría por subgrupos

### Pregunta 2.1

Separe 70 % para entrenar y 30 % para probar, estratificado por incumple y con semilla 2026. Entrene una regresión logística con pesos balanceados y estandarización, sin sexo, edad ni estado civil. Reporte ROC-AUC, recall, precisión y tasa de falsos positivos con umbral 0,50, y la tasa base de incumplimiento. ¿Por qué la exactitud no basta?

In [9]:
# Escriba aquí el código de su equipo y documente cada decisión con comentarios.

**Respuesta:**

### Pregunta 2.2

Audite por sexo, grupo de edad (21-29, 30-44 y 45 o más) y estado civil. Por subgrupo reporte personas, tasa de positivos predichos, recall, FPR y FNR, omitiendo grupos de menos de 100 personas. ¿Qué subgrupo recibe más alertas y cuál tiene más falsos positivos?

In [10]:
# Escriba aquí el código de su equipo y documente cada decisión con comentarios.

**Respuesta:**

### Pregunta 2.3

Cuantifique las brechas (valor más alto menos valor más bajo) de cada métrica y dimensión. Priorice una brecha con tres criterios: magnitud, tamaño de los grupos y tipo de error, y explique por qué una brecha no prueba por sí sola discriminación.

In [11]:
# Escriba aquí el código de su equipo y documente cada decisión con comentarios.

**Respuesta:**

### Pregunta 2.4

Compruebe si hay proxies: entrene un modelo simple que prediga si el cliente tiene 45 años o más usando solo las variables del modelo y reporte su ROC-AUC en la prueba. ¿Qué indica ese valor sobre excluir la edad del entrenamiento?

In [12]:
# Escriba aquí el código de su equipo y documente cada decisión con comentarios.

**Respuesta:**

> **Criterio de aceptación:** la partición, el modelo y el umbral se fijan antes de auditar, las brechas se sustentan con el tamaño del grupo y no se cambian los parámetros para reducirlas.

# Ejercicio 3 — Explicabilidad, controles y hallazgos

### Pregunta 3.1

Calcule la importancia global con SHAP para una muestra de 500 clientes de la prueba (semilla 2026). Presente las cinco variables con mayor contribución y explique qué significa que una variable tenga mayor contribución sin usar lenguaje causal.

In [13]:
# Escriba aquí el código de su equipo y documente cada decisión con comentarios.

**Respuesta:**

### Pregunta 3.2

Elija un falso positivo y un falso negativo de la prueba y explique con SHAP qué variables aumentaron o redujeron la probabilidad en cada caso. ¿Qué advertencia debe hacerse sobre la lectura de estas explicaciones?

In [14]:
# Escriba aquí el código de su equipo y documente cada decisión con comentarios.

**Respuesta:**

### Pregunta 3.3

Complete el checklist de ocho controles (finalidad, minimización, calidad, linaje, acceso, transparencia, auditoría de subgrupos y supervisión humana) con un estado (Cumple, Parcial, No evidenciado o No aplica) y la evidencia que lo sustenta. ¿Qué controles no pueden demostrarse con código?

In [15]:
# Escriba aquí el código de su equipo y documente cada decisión con comentarios.

**Respuesta:**

### Pregunta 3.4

Registre al menos tres hallazgos con severidad (crítico, alto, medio o bajo), evidencia, acción, responsable y estado. Indique cuál impediría el uso del modelo hasta ser resuelto y por qué.

In [16]:
# Escriba aquí el código de su equipo y documente cada decisión con comentarios.

**Respuesta:**

> **Criterio de aceptación:** la explicación no se presenta como causalidad, el checklist se completa con evidencia y no como certificación legal, y cada hallazgo tiene responsable y acción verificable.

# Ejercicio 4 — Asistente Text-to-SQL: construcción y seguridad

### Pregunta 4.1

Cree la tabla credito en DuckDB, guárdela en un archivo y ábrala en solo lectura con un tope de 1 000 filas. Redacte el diccionario de datos (con los códigos de sex, education, marriage y pay_) y el contexto que recibirá el LLM. ¿Por qué incluir los códigos reduce errores?

In [17]:
# Escriba aquí el código de su equipo y documente cada decisión con comentarios.

**Respuesta:**

### Pregunta 4.2

Conecte generate_sql() con el proveedor de LLM del curso y registre proveedor y modelo. Programe un validador que bloquee todo lo que no sea una sola sentencia SELECT o WITH y escriba al menos seis pruebas, entre ellas una que muestre que una cadena entre comillas no genera un falso bloqueo.

In [18]:
# Escriba aquí el código de su equipo y documente cada decisión con comentarios.

**Respuesta:**

### Pregunta 4.3

Escriba la consulta de referencia de las ocho preguntas respondibles y reporte sus resultados: (1) ¿cuántos clientes hay?; (2) ¿cuál es la tasa de incumplimiento global?; (3) ¿cuál es el límite de crédito promedio?; (4) ¿cuántos clientes tienen más de 45 años?; (5) ¿cuál es la tasa de incumplimiento por sexo?; (6) ¿cuántos clientes tienen un estado de pago pay_1 mayor o igual a 2?; (7) ¿cuál es el monto promedio de la factura bill_amt1 de quienes incumplieron?; (8) ¿qué grupo de edad (21-29, 30-44, 45 o más) tiene la mayor tasa de incumplimiento?

In [19]:
# Escriba aquí el código de su equipo y documente cada decisión con comentarios.

**Respuesta:**

### Pregunta 4.4

Dos preguntas más no deben responderse sin aclaración: (9) ¿cuántos clientes de alto riesgo hay? y (10) ¿cuál es el ingreso promedio de los clientes? Defina el comportamiento esperado del asistente para cada una y explique por qué ejecutar una consulta inventada sería más riesgoso que no responder.

In [20]:
# Escriba aquí el código de su equipo y documente cada decisión con comentarios.

**Respuesta:**

> **Criterio de aceptación:** la clave no aparece en el notebook, la seguridad se impone con validador y base de solo lectura y no solo con el prompt, y las referencias se calculan con los mismos datos que usará el asistente.

# Ejercicio 5 — Evaluación del asistente y mitigación

### Pregunta 5.1

Evalúe las diez preguntas: registre la respuesta del asistente, el SQL limpio, el estado de seguridad, la ejecución, la coincidencia con la referencia (en las ocho respondibles) y el tipo de falla (esquema, sintaxis o ejecución, semántica, seguridad o formato).

In [21]:
# Escriba aquí el código de su equipo y documente cada decisión con comentarios.

**Respuesta:**

### Pregunta 5.2

Calcule exactitud de ejecución, tasa de alucinación de esquema, tasa de error semántico y tasa de bloqueo sobre las ocho preguntas respondibles, y diga si el asistente se comportó como corresponde en las preguntas 9 y 10.

In [22]:
# Escriba aquí el código de su equipo y documente cada decisión con comentarios.

**Respuesta:**

### Pregunta 5.3

Analice las fallas y proponga una mitigación verificable para cada tipo. Aplique una sola mitigación al contexto, repita la batería completa y compare las tasas antes y después. ¿Mejoró? Informe el resultado tal como sale.

In [23]:
# Escriba aquí el código de su equipo y documente cada decisión con comentarios.

**Respuesta:**

### Pregunta 5.4

Describa un error semántico concreto (real o posible) y qué decisión de crédito equivocada podría causar si nadie lo detecta. ¿Qué control lo habría detectado?

In [24]:
# Escriba aquí el código de su equipo y documente cada decisión con comentarios.

**Respuesta:**

> **Criterio de aceptación:** la batería se evalúa completa antes y después de la mitigación con el mismo procedimiento, no se descartan preguntas difíciles y las fallas se clasifican con evidencia.

# Ejercicio 6 — Decisión integrada y sensibilidad

### Pregunta 6.1

Escriba una regla reproducible para cada herramienta y aplíquela sin cambiarla. Cada regla debe indicar el criterio, el dato que la decide y la acción. Ejemplo para el modelo: usarlo con condiciones si la brecha de FPR entre grupos de edad es menor que 0,10 y no hay hallazgos críticos abiertos. Defina la regla del asistente y diga qué recomendación produce cada una.

In [25]:
# Escriba aquí el código de su equipo y documente cada decisión con comentarios.

**Respuesta:**

### Pregunta 6.2

Sensibilidad del modelo: repita la auditoría por grupo de edad con umbrales de 0,30, 0,50 y 0,70 y reporte recall, FPR y brecha de FPR. Sensibilidad del asistente: compare las tasas antes y después de la mitigación. ¿Cuándo cambia cada decisión?

In [26]:
# Escriba aquí el código de su equipo y documente cada decisión con comentarios.

**Respuesta:**

### Pregunta 6.3

Si el banco solo pudiera aprobar una de las dos herramientas el próximo trimestre, ¿cuál sustentaría con más solidez? Compare, con cifras propias, el tamaño del riesgo, la estabilidad de la evidencia y la sensibilidad a los supuestos.

In [27]:
# Escriba aquí el código de su equipo y documente cada decisión con comentarios.

**Respuesta:**

> **Criterio de aceptación:** distingue evidencia, supuestos y tolerancia al riesgo; aplica las reglas sin cambiarlas para obtener una conclusión preferida.

# Preguntas escritas e informe de recomendación

Las interpretaciones de los seis ejercicios forman parte de sus criterios técnicos. El cierre siguiente aporta la evidencia del criterio de lectura crítica y límites declarados.

### Pregunta A

¿Por qué un buen ROC-AUC no basta para aprobar el uso de un modelo? ¿Qué evidencia adicional exigió y qué mostró la auditoría por subgrupos?

**Respuesta:**

### Pregunta B

¿Por qué la seguridad del asistente no puede depender solo de lo que se le pide en el prompt? ¿Qué controles impuso fuera del modelo?

**Respuesta:**

### Pregunta C

¿Qué podría concluirse equivocadamente mirando solo la exactitud global del modelo o solo la tasa de acierto del asistente? Sustente su respuesta con los resultados obtenidos.

**Respuesta:**

## Informe de recomendación

Redacte un máximo de 180 palabras dirigido a la dirección del área de riesgos. Recomiende, para cada herramienta, usar, usar con condiciones o no usar todavía. Incluya al menos cuatro cifras propias: ROC-AUC del modelo, la mayor brecha auditada, la exactitud del asistente y su tasa de alucinación. Declare el principal riesgo, una condición de cambio y dos limitaciones.

In [28]:
# Pegue aquí su informe (máximo 180 palabras) y ejecute la celda para contar las palabras.
informe = """
Escriba el informe de recomendación dirigido a la dirección del área de riesgos.
"""
print("Palabras:", len(informe.split()))

Palabras: 13


## Advertencia metodológica

Las métricas del modelo se calculan sobre una sola partición de datos históricos de un banco de Taiwán de 2005 y no se extrapolan a otros mercados ni al 2026. Una brecha entre grupos no prueba por sí sola discriminación, y SHAP describe cómo se comporta el modelo, no las causas del incumplimiento. El checklist es académico: no constituye certificación legal.

Los resultados del asistente dependen del proveedor y del modelo de LLM elegidos y pueden variar entre ejecuciones y versiones. Una tasa de acierto alta no elimina la necesidad de controles, y diez preguntas son una muestra pequeña. Una buena métrica no garantiza una buena decisión: las reglas se fijan antes de ver la evaluación final y el resultado se informa tal como sale.

# Rúbrica de la evaluación (resumen)

| Criterio | Ejercicios | Puntos |
|---|---|---|
| 1. Ficha de datos y gobierno | 1 | 2 |
| 2. Auditoría del modelo | 2 y 3 | 4 |
| 3. Asistente Text-to-SQL y seguridad | 4 y 5 | 4 |
| 4. Decisión integrada y lectura crítica | 6 y cierre | 3 |
| 5. Documentación y código propio | todos | 3 |
| **Total grupal** | | **16** |

Se suman 4 puntos de actividades individuales de las semanas 13 y 14. **Descuentos sobre el puntaje grupal:** puntualidad y dedicación (1), ortografía y redacción de las conclusiones (1), notebook que no ejecuta de principio a fin (2), uso de IA no declarado (2), código, comentarios o respuestas que no están redactados con palabras propias del equipo o integrante que no puede explicar el trabajo (**5 como mínimo**).

# Fuente de datos

Yeh, I.-C. (2009). *Default of Credit Card Clients* [Conjunto de datos]. UCI Machine Learning Repository. https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients

Referencias de estudio: DAMA International, *DAMA-DMBOK*; Molnar, C., *Interpretable Machine Learning*; Lundberg, S. y Lee, S.-I. (2017), *A Unified Approach to Interpreting Model Predictions*; documentación de DuckDB.